Ingesta Hermética y Replicación del Orquestador BPO

El primer requerimiento para asegurar la validez técnica de este análisis cruzado es replicar exactamente las condiciones del entorno clásico (Notebook 02). 

Procedemos a importar el tensor particionado `train_set_v7.parquet` originado en la Fase 1, recuperando el vector `fold_id` para congelar la estratificación de los datos. A continuación, reinstanciamos el orquestador BPO (`evaluar_modelo_bpo`). El algoritmo de evaluación ha sido ajustado internamente en sus índices lógicos para tolerar el ingreso de matrices densas de NumPy (los futuros *embeddings*) en lugar de exclusivas estructuras de Pandas, manteniendo intacto el cálculo matemático de la matriz de contingencia (Log-Loss, F1) y la restricción financiera (umbral estático del 0.60).

In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.metrics import log_loss, f1_score, cohen_kappa_score
import warnings

# Bloqueamos el spam estético en la salida estándar
warnings.filterwarnings('ignore', category=UserWarning)

print("Cargando matriz Gold...")
# El motor fastparquet elude fugas de memoria en C++ que suele provocar pyarrow en Windows
df_train = pd.read_parquet('../data/gold/train_set_v7.parquet', engine='fastparquet')

# Aislamiento de vectores estáticos globales
X = df_train['full_text']
y = df_train['target_tripleta']
folds = df_train['fold_id']

print(f"Dataset cargado -> Volumen: {len(df_train)} tickets | Clases: {y.nunique()}")

def evaluar_modelo_bpo(estimator, X, y, fold_ids, umbral_confianza=0.60):
    resultados = []
    
    for fold in sorted(fold_ids.unique()):
        # Vectorizamos la máscara para evitar colisiones entre el index de Pandas y los arrays puros de Numpy
        idx_train = (fold_ids != fold).values
        idx_val = (fold_ids == fold).values
        
        # Indexación robusta (polimorfismo). Soporta Series de Pandas o tensores NumPy
        X_train = X.loc[idx_train] if hasattr(X, 'loc') else X[idx_train]
        X_val = X.loc[idx_val] if hasattr(X, 'loc') else X[idx_val]
        
        y_train = y.loc[idx_train]
        y_val = y.loc[idx_val]
        
        # Telemetría de Entrenamiento (Coste computacional)
        t0 = time.time()
        estimator.fit(X_train, y_train)
        fit_time = time.time() - t0
        
        # Telemetría de Inferencia (Latencia operativa)
        t1 = time.time()
        y_proba = estimator.predict_proba(X_val)
        latency_ms = ((time.time() - t1) / len(X_val)) * 1000
        
        y_max_proba = np.max(y_proba, axis=1)
        y_pred_bruto = estimator.classes_[np.argmax(y_proba, axis=1)]
        
        # Mapeo de Pérdida Académica (Corregido zero_division)
        ll = log_loss(y_val, y_proba, labels=estimator.classes_)
        f1_mac = f1_score(y_val, y_pred_bruto, average='macro', zero_division=0)
        f1_wei = f1_score(y_val, y_pred_bruto, average='weighted', zero_division=0)
        kappa = cohen_kappa_score(y_val, y_pred_bruto)
        
        # Motor Lógico BPO (El filtro Auditor)
        mask_auto = y_max_proba >= umbral_confianza
        tasa_auto = np.mean(mask_auto)
        
        if np.sum(mask_auto) > 0:
            y_val_auto = y_val.values[mask_auto]
            y_pred_auto = y_pred_bruto[mask_auto]
            prec_cond = np.mean(y_val_auto == y_pred_auto)
        else:
            prec_cond = 0.0
            
        resultados.append({
            'Fold': fold,
            'Fit_Time_s': fit_time,
            'Latency_ms': latency_ms,
            'Log_Loss': ll,
            'F1_Macro': f1_mac,
            'F1_Weighted': f1_wei,
            'Kappa': kappa,
            'Tasa_Automatizacion': tasa_auto,
            'Precision_Condicionada': prec_cond
        })
        
    return pd.DataFrame(resultados)

print("Orquestador BPO (Motor de Evaluación) en memoria.")

Cargando matriz Gold...
Dataset cargado -> Volumen: 23720 tickets | Clases: 102
Orquestador BPO (Motor de Evaluación) en memoria.


Extracción Estática de Embeddings (Frozen Transformer)

Abandonamos definitivamente el espacio de 15.000 dimensiones discretas del TF-IDF para proyectar la semántica de los tickets en un espacio vectorial continuo y denso de 384 dimensiones. 

Para lograrlo, inyectamos el modelo neuronal pre-entrenado `BAAI/bge-small-en-v1.5` operando estrictamente como extractor de características (*Frozen*). En base a las directivas de control de calidad (QA), forzamos la máxima ventana de contexto del modelo (`max_length=512` tokens) para garantizar que el modelo lea el problema real del cliente, sorteando el ruido introducido por firmas automáticas y saludos. Dado que los pesos de la red neuronal no van a ser actualizados (*no hay retropropagación ni Fine-Tuning*), la proyección estática global de los datos antes del bucle de validación cruzada no infringe las reglas de *Data Leakage* y ahorra horas de inferencia redundante en la CPU. Adicionalmente, forzamos la descarga del modelo dentro del entorno local del proyecto para garantizar una arquitectura de despliegue offline y autocontenida.

In [2]:
from sentence_transformers import SentenceTransformer
import time

print("Instanciando el modelo neuronal BAAI/bge-small-en-v1.5 (Aislamiento Local y Control CPU)...")
modelo_encoder = SentenceTransformer(
    'BAAI/bge-small-en-v1.5', 
    cache_folder='../models/bge-small_local/',
    device='cpu'
)

modelo_encoder.max_seq_length = 512

print(f"Iniciando extracción vectorial para {len(X)} registros (Carga CPU Intensiva)...")
t0_embed = time.time()

# PURIFICACIÓN DEFENSIVA DE ÚLTIMA MILLA
# Blindamos el tensor contra corrupciones de lectura del .parquet (Nulos fantasma)
X_limpio = X.fillna("").astype(str).tolist()

# Proyección estática y conversión a hiperplano denso sobre la lista purificada
X_embeddings = modelo_encoder.encode(X_limpio, show_progress_bar=True)

tiempo_total = time.time() - t0_embed
print(f"Proyección semántica completada en {tiempo_total/60:.2f} minutos.")
print(f"Morfología matemática del nuevo tensor: {X_embeddings.shape}")

d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Instanciando el modelo neuronal BAAI/bge-small-en-v1.5 (Aislamiento Local y Control CPU)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1813.40it/s]


Iniciando extracción vectorial para 23720 registros (Carga CPU Intensiva)...


Batches: 100%|██████████| 742/742 [01:47<00:00,  6.93it/s]


Proyección semántica completada en 1.79 minutos.
Morfología matemática del nuevo tensor: (23720, 384)


Modelo C: Regresión Logística sobre Espacio Semántico Denso (Embeddings)

Tras comprimir la dispersión léxica de 15.000 dimensiones a un hiperplano continuo de 384 dimensiones semánticas, reinstanciamos el clasificador lineal. 

La Regresión Logística, que colapsó en la Fase 2 debido a la naturaleza ortogonal del TF-IDF (Tasa de Automatización < 1%), se somete ahora a un entorno de alta densidad neuronal. Al reducir drásticamente los gradientes a optimizar (de 15k a 384 por clase), el solucionador cuasi-Newton por defecto (`lbfgs`) vuelve a ser viable y extremadamente rápido en memoria local. Se mantiene la regularización L2 y la penalización de clases (`class_weight='balanced'`) para garantizar que el hiperplano castigue severamente el error sobre el *long tail* de la operativa BPO. El objetivo es auditar empíricamente si la reestructuración geométrica inducida por el Transformer permite a la regresión emitir curvas de probabilidad que superen la barrera estática de confianza del 0.60.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

print("Instanciando Modelo C (Pipeline: Escalado Estándar + LogReg Densa)...")

# La magia matemática: StandardScaler expande la anisotropía de los tensores
# para que el solucionador lbfgs encuentre gradientes viables.
pipeline_semantico = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        class_weight='balanced', 
        max_iter=1000, 
        solver='lbfgs',
        random_state=42,
        n_jobs=-1
    )
)

print("Iniciando orquestación de Cross-Validation sobre la matriz neuronal...")
df_resultados_semanticos = evaluar_modelo_bpo(
    estimator=pipeline_semantico, 
    X=X_embeddings, 
    y=y, 
    fold_ids=folds, 
    umbral_confianza=0.60
)

print("\n--- Telemetría Consolidada: Modelo C Corregido ---")
metricas_promedio_semantico = df_resultados_semanticos.mean().drop('Fold')
print(metricas_promedio_semantico.round(4))

Instanciando Modelo C (Pipeline: Escalado Estándar + LogReg Densa)...
Iniciando orquestación de Cross-Validation sobre la matriz neuronal...


d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
d:\MasterEvolve\Proyecto TFM\SITOR\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.


--- Telemetría Consolidada: Modelo C Corregido ---
Fit_Time_s                7.7326
Latency_ms                0.0027
Log_Loss                  4.8509
F1_Macro                  0.0394
F1_Weighted               0.0444
Kappa                     0.0226
Tasa_Automatizacion       0.0809
Precision_Condicionada    0.2558
dtype: float64
